# Part 2: Computer Vision Problem Formulation and CNN Prototype

**Goal:** Build a CNN-based image classifier using a real image dataset  
**Dataset Source:** [Part 2 Dataset – Google Drive](https://drive.google.com/drive/folders/1akV6po4Nrgkc3yQrJkzA6cJlV-wBvUYs?usp=sharing)


## Setup – Mount Drive & Install Libraries

In [ ]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install/import all required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print(f"TensorFlow version: {tf.__version__}")

# Create output directories
os.makedirs('results', exist_ok=True)
os.makedirs('sample_predictions', exist_ok=True)
print("Output directories created.")


## Task 1: Problem Identification

This dataset represents an **Image Classification** problem.

**Why Image Classification?**
- Each image belongs to exactly one category/class (mutually exclusive labels)
- The goal is to assign a single class label to each input image
- No bounding boxes or pixel-level masks are needed
- The CNN learns to map raw pixel values → class probabilities via a softmax output layer

This is the most fundamental computer vision task, and CNNs are the standard approach for it.


## Task 2: Dataset Exploration

In [ ]:
# ── SET YOUR DATASET PATH HERE ──
# After mounting Drive, find your dataset folder path and set it below
# Example: '/content/drive/MyDrive/Part2_Dataset'

import os

# Auto-detect dataset path from Drive
DRIVE_BASE = '/content/drive/MyDrive'

# List folders to help locate the dataset
print("Folders in MyDrive:")
for f in os.listdir(DRIVE_BASE):
    print(f" -", f)


In [ ]:
# Set dataset path (update this after running the cell above)
# Look for a folder that contains subfolders (each subfolder = one class)
DATASET_PATH = '/content/drive/MyDrive/Part2_Dataset'  # <-- UPDATE THIS PATH

# Explore dataset structure
classes = sorted([d for d in os.listdir(DATASET_PATH)
                  if os.path.isdir(os.path.join(DATASET_PATH, d))])

print(f"Number of classes: {len(classes)}")
print(f"Classes: {classes}")
print()

class_counts = {}
for cls in classes:
    cls_path = os.path.join(DATASET_PATH, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
    class_counts[cls] = len(imgs)
    print(f"  {cls}: {len(imgs)} images")

print(f"\nTotal images: {sum(class_counts.values())}")


In [ ]:
# Check image dimensions
import cv2
from PIL import Image

sample_dims = []
for cls in classes:
    cls_path = os.path.join(DATASET_PATH, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        img = Image.open(os.path.join(cls_path, imgs[0]))
        sample_dims.append((cls, img.size, img.mode))
        print(f"  {cls}: size={img.size}, mode={img.mode}")


In [ ]:
# Plot class distribution (check imbalance)
plt.figure(figsize=(10, 5))
bars = plt.bar(class_counts.keys(), class_counts.values(),
               color=plt.cm.Set2(range(len(classes))), edgecolor='black')
for bar, val in zip(bars, class_counts.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', fontweight='bold')
plt.title('Number of Images per Class', fontsize=14)
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150)
plt.show()


In [ ]:
# Display sample images from each class
n_classes = len(classes)
fig, axes = plt.subplots(n_classes, 5, figsize=(15, 3 * n_classes))
if n_classes == 1:
    axes = [axes]

for i, cls in enumerate(classes):
    cls_path = os.path.join(DATASET_PATH, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:5]
    for j, img_name in enumerate(imgs):
        img = Image.open(os.path.join(cls_path, img_name))
        ax = axes[i][j] if n_classes > 1 else axes[j]
        ax.imshow(img)
        ax.axis('off')
        if j == 0:
            ax.set_title(cls, fontsize=11, fontweight='bold')

plt.suptitle('Sample Images per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()


## Task 3: Image Preprocessing

In [ ]:
# Configuration
IMG_SIZE   = (128, 128)   # Resize all images to 128x128
BATCH_SIZE = 32
SEED       = 42

# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,           # Normalize pixel values to [0, 1]
    validation_split=0.2,           # 80% train, 20% validation
    rotation_range=20,              # Augmentation: random rotation
    width_shift_range=0.15,         # Augmentation: horizontal shift
    height_shift_range=0.15,        # Augmentation: vertical shift
    horizontal_flip=True,           # Augmentation: flip
    zoom_range=0.15,                # Augmentation: zoom
    fill_mode='nearest'
)

# Test data generator — only rescale, no augmentation
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

# Load training set
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True
)

# Load validation set
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

NUM_CLASSES = len(train_generator.class_indices)
print(f"\nClass indices: {train_generator.class_indices}")
print(f"Training batches  : {len(train_generator)}")
print(f"Validation batches: {len(val_generator)}")
print(f"Number of classes : {NUM_CLASSES}")


## Task 4: CNN Model Creation

In [ ]:
def build_cnn(num_classes, img_size=(128, 128), learning_rate=0.001):
    """
    Build a CNN with:
    - Conv layers (feature extraction)
    - ReLU activations (non-linearity)
    - MaxPooling (spatial downsampling)
    - BatchNormalization (stable training)
    - Flatten + Dense layers (classification)
    - Softmax output (multi-class probabilities)
    """
    model = keras.Sequential([
        # Block 1 – learn low-level features (edges, textures)
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=(img_size[0], img_size[1], 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 2 – learn mid-level features (shapes, patterns)
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3 – learn high-level features
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Flatten & classify
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')  # output layer
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_cnn(NUM_CLASSES, IMG_SIZE)
model.summary()


## Task 5: Model Training and Evaluation

In [ ]:
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10,
                           restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                               patience=5, min_lr=1e-6, verbose=1)

# Train the model
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:
# Plot accuracy & loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history.history[metric],     label=f'Train {title}', linewidth=2)
    ax.plot(history.history[f'val_{metric}'], label=f'Val {title}',
            linewidth=2, linestyle='--')
    ax.set_title(f'CNN – {title} Curve', fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/accuracy_loss_curves.png")


In [ ]:
# Evaluate on validation set
val_loss, val_acc = model.evaluate(val_generator, verbose=0)
print(f"Validation Loss    : {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc*100:.2f}%")

# Predictions
val_generator.reset()
y_pred_prob = model.predict(val_generator, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = val_generator.classes

class_names = list(train_generator.class_indices.keys())

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(6, NUM_CLASSES), max(5, NUM_CLASSES - 1)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(colorbar=False, cmap='Blues', xticks_rotation=30)
plt.title('Confusion Matrix – CNN Model', fontsize=14)
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/confusion_matrix.png")


In [ ]:
# Sample predictions on test images
val_generator.reset()
images, labels = next(val_generator)
preds = model.predict(images[:12], verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i in range(12):
    axes[i].imshow(images[i])
    true_label = class_names[np.argmax(labels[i])]
    pred_label = class_names[np.argmax(preds[i])]
    conf       = np.max(preds[i]) * 100
    color = 'green' if true_label == pred_label else 'red'
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.1f}%)",
                      color=color, fontsize=9)
    axes[i].axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sample_predictions/prediction_outputs.png")


## Task 6: CNN Concept Explanation

### What is Convolution?
Convolution is a mathematical operation where a small matrix called a **filter** (or kernel) slides across the input image and computes dot products at each position. This produces a **feature map** that highlights specific patterns — such as edges, corners, or textures — at each spatial location. Different filters learn to detect different visual features automatically during training.

### Why is Pooling Used?
Pooling (typically **MaxPooling**) reduces the spatial dimensions of feature maps by keeping only the most prominent value in each local region. This serves two purposes: it reduces computation and memory, and it makes the model **translation-invariant** — meaning it can recognize a pattern (like an eye) regardless of where it appears in the image.

### Why is ReLU Commonly Used in CNNs?
**ReLU** (`f(x) = max(0, x)`) introduces non-linearity, allowing the network to learn complex visual patterns that a linear model cannot. It is preferred in CNNs because it does not suffer from the vanishing gradient problem (unlike sigmoid/tanh), it is computationally fast, and it produces sparse activations which help the network focus on the most relevant features.

### Why are CNNs Better Than Feed-Forward Networks for Images?
- **Parameter sharing:** A filter's weights are shared across the entire image, drastically reducing parameters compared to a fully connected layer.
- **Local connectivity:** Each neuron only connects to a small region, capturing local spatial structure naturally.
- **Translation invariance:** Pooling makes CNNs robust to small shifts in object position.
- A 128×128 RGB image has 49,152 pixels — a fully connected layer would need millions of parameters, while a CNN handles it efficiently with a few thousand shared weights per filter.

---

## Task 7: Business Use Case Mapping

### Domain: Healthcare – Medical Image Classification

A CNN model similar to the one built here can be deployed in **radiology and pathology** workflows to automatically classify medical images such as X-rays, MRI scans, or histology slides.

**Example:** Classifying chest X-rays into categories — Normal, Pneumonia, COVID-19, or Tuberculosis.

**Business Benefits:**
- **Speed:** A trained CNN classifies an image in milliseconds, far faster than manual review.
- **Consistency:** Eliminates human fatigue and inter-rater variability between radiologists.
- **Scalability:** Can process thousands of scans per hour in high-volume hospitals or screening programs.
- **Early detection:** Subtle patterns invisible to the human eye can be detected reliably, improving patient outcomes.

This use case directly saves lives and reduces diagnostic costs — making computer vision one of the highest-impact AI applications in healthcare.
